In [ ]:
############################################################
#read in setup -- kind of stupid system actually
############################################################

import pandas as pd
from datetime import datetime, timedelta






with open('setup.py') as f:
    code = f.read()
exec(code)
run_sql("use hoodalgo_db")












#Create/reset driver
#################################################################################################################
try:
    driver.close()
except:
    pass

try:
    driver.quit()
except:
    pass

import time, os

time.sleep(.05)

# SAFE CLEAN (won’t kill your real Chrome)
os.system("pkill -f chromedriver")
os.system("pkill -f 'chrome.*--remote-debugging-port'")
os.system("pkill -f 'chrome.*--user-data-dir=/Users/deanemarks/selenium_chrome_profile'")

time.sleep(.05)

# Recreate driverD
driver = create_driver_profile_1()

time.sleep(.05)

# Load page
file_path = "file://" + base_dir + "templates/loading_page.html"
driver.get(file_path)

driver.execute_script("document.body.style.zoom='100%'")
time.sleep(2)
#################################################################################################################










#EXECUTE in MODUELS/ FUNCTIONS -  MAKES SHARED NAMESPACE
############################################################
with open('functions.py') as f:
    code = f.read()
exec(code)
############################################################
















#Update Robinhood Universe Every Week
#################################################################################################################
file_path = 'robinhood_universe.csv'
created_ts = os.path.getctime(file_path)
universe_created_dt = datetime.fromtimestamp(created_ts)
one_week_ago = datetime.now() - timedelta(days=7)


if universe_created_dt > one_week_ago:
    number_of_days = str(universe_created_dt - datetime.now()).split(",")[0].replace('-','').replace(' day','')
    update_loading_page(driver, f"Robinhood Universe Scraped {number_of_days} Day Ago - Dont Rescrape")
    time.sleep(1.5)


if universe_created_dt < one_week_ago:
    update_loading_page(driver, "Scraping Full Robinhood Equities Universe")
    scrape_robinhood_universe(num_workers = 20)
    update_loading_page(driver, "Robinhood Equities Universe successfully Scrapped")
    time.sleep(1.5)
#################################################################################################################
















#Fetch Fundamentals Data and filter every 10 minutes: 
#################################################################################################################

update_fundamentals_time_delta = 3


file_path = base_dir + "robinhood_universe_filtered.csv"
created_ts = os.path.getctime(file_path)
filtered_universe_ran_time = datetime.fromtimestamp(created_ts)
filtered_universe_run_again_time = filtered_universe_ran_time + timedelta(minutes=update_fundamentals_time_delta)




if os.path.exists(file_path):


    #If PAST  RUN_AGAIN_TIME example 
    if datetime.now() > filtered_universe_run_again_time:
        print(f'not within the {update_fundamentals_time_delta} minute window/n scraping all tickers then will refilter')

        
        df = pd.read_csv('robinhood_universe.csv')
        stock_universe_df = df[
            (df['tradeable'] == True) &
            (df['state'] == 'active') &
            (df['type'] == 'stock') &  # 👈 this is what you're missing
            #(df['all_day_tradability'] == 'tradable') &
            (df['fractional_tradability'] == 'tradable')
        ]
        stock_universe_list = stock_universe_df['symbol'].to_list()
        
        #UPDATE GUI NOTE
        update_loading_page(driver, f"NOW > RUN_AGAIN_TIME - Rescrape full Robinhood_universe  ")
        time.sleep(1.1)
        update_loading_page(driver, f"Fetching Robinhood Fundamentals Data - {len(stock_universe_list)} stocks ")


        
        #SCRAPE FUNDAMENTASL FX 
        robinhood_fundamentals_df = fetch_robinhood_fundamentals(ticker_list = stock_universe_list,chunk_size = 100)

        
        #UPDATE GUI NOTE
        update_loading_page(driver, "Scrape Complete ")
        time.sleep(1)


        #Fundamental Filter (DO NOT OVERFILTER HERE)
        ####################
        
        #Pre-Calculate Metrics
        df = robinhood_fundamentals_df.copy()
        df['average_volume_30_days'] = df['average_volume_30_days'].replace(0, 1)
        df['open'] = df['open'].replace(0, 1)
        
        # NO rel_volume filter here
        # NO range_pct filter here
        
        fundamentals_filter_df = df[
        
            # =========================
            # PRICE RANGE (stable)
            # =========================
            (df['open'] >= 1.5) &
            (df['open'] <= 30) &
        
            # =========================
            # SIZE (stable)
            # =========================
            (df['market_cap'] >= 20_000_000) &
            (df['market_cap'] <= 10_000_000_000) &
        
            # =========================
            # FLOAT (stable)
            # =========================
            (df['shares_float'] >= 1_000_000) &
        
            # =========================
            # BASE LIQUIDITY (loose)
            # =========================
            (df['average_volume_30_days'] >= 150_000)
        
        ]
        ####################

        
        #Filter tickers update load page
        filtered_tickers = fundamentals_filter_df['ticker'].dropna().unique().tolist()
        update_loading_page(driver, f"Done - {len(filtered_tickers)} stocks passed Fundamental filter")
        time.sleep(1)
        fundamentals_filter_df.to_csv("robinhood_universe_filtered.csv")






    
    
        
    #If INSIDE RUN_AGAIN TIME 
    if datetime.now() < filtered_universe_run_again_time:
        print('is within the 30 minute window/n scraping only the filtered tickers then will do all after 30 minutes')

        filtered_universe_df = pd.read_csv("robinhood_universe_filtered.csv")
        filtered_universe_list = filtered_universe_df['ticker'].to_list()



        #UPDATE GUI NOTE
        update_loading_page(driver, f"NOW < RUN_AGAIN_TIME Scraping filtered_robinhood_universe  ")
        time.sleep(1.1)
        update_loading_page(driver, f"Fetching Robinhood Fundamentals Data - {len(filtered_universe_list)} stocks ")


        

        #filter the tickers here. 
        robinhood_fundamentals_df = fetch_robinhood_fundamentals(ticker_list = filtered_universe_list,chunk_size = 100)
        filtered_tickers = robinhood_fundamentals_df['ticker'].dropna().unique().tolist()
        update_loading_page(driver, f"Complete ")

        

    
else:
    print("File does NOT exist")
    robinhood_fundamentals_df = fetch_robinhood_fundamentals(ticker_list = stock_universe_list,chunk_size = 100)
    robinhood_fundamentals_df.to_csv("robinhood_universe_filtered.csv")
    print('saved')


#################################################################################################################




    










In [ ]:
# Optional dev limiter. Set to an integer while testing; leave as None for the full run.
MAX_TICKERS = None

if MAX_TICKERS is not None:
    filtered_tickers = filtered_tickers[:MAX_TICKERS]

len(filtered_tickers)


In [ ]:
# Stocktwits results are created in the Selenium scrape cell below.
stocktwits_signal_df = pd.DataFrame()


In [ ]:
#Scrape Stocktwits W Selenium Threads (5 Workers, DF Safe)
#############################################################################################################


start = time.time()

import os
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

# --- HARD RESET (light) ---
try:
    os.system("pkill -f chromedriver")
except:
    pass

time.sleep(1)


# --- CONFIG ---
num_workers = 5


# --- SPLIT TICKERS ---
ticker_lists = [filtered_tickers[i::num_workers] for i in range(num_workers)]


# --- CREATE DRIVERS ---
drivers = []

i = 0
while i < num_workers:

    try:
        profile_name = f"selenium_chrome_profile_stocktwits_{i+1}"
        stocktwits_driver = create_driver_profile(profile_name)
        drivers.append(stocktwits_driver)
    except:
        drivers.append(None)

    i += 1


# --- RUN THREADS ---
dfs = []

with ThreadPoolExecutor(max_workers=num_workers) as executor:

    futures = []

    i = 0
    while i < num_workers:

        if drivers[i] is not None:
            futures.append(
                executor.submit(
                    selenium_fetch_stocktwits_sentiment,
                    ticker_lists[i],
                    drivers[i]
                )
            )

        i += 1

    # collect DATAFRAMES (NOT extend)
    i = 0
    while i < len(futures):

        result = futures[i].result()

        if result is not None and len(result) > 0:
            dfs.append(result)   # 👈 APPEND DF

        i += 1


# --- CLEANUP ---
i = 0
while i < len(drivers):

    try:
        if drivers[i]:
            drivers[i].quit()
    except:
        pass

    i += 1


# --- FINAL COMBINE ---
if len(dfs) > 0:
    final_df = pd.concat(dfs, ignore_index=True)
else:
    final_df = pd.DataFrame()



end = time.time()
scrape_time = end-start
print(scrape_time)



#############################################################################################################


In [ ]:
stocktwits_signal_df = final_df.copy()
stocktwits_signal_df.sort_values(by="sent_score", ascending=False).reset_index(drop=True).head(50)


In [ ]:
# Google News booster. This should boost tickers with fresh article volume, not remove everything else.
##################################################################################################################
update_loading_page(driver, "Fetching Google News (API Version)")
time.sleep(1)
update_loading_page(driver, f"Fetching Google News (API Version) - {len(filtered_tickers)} stocks ")


# Scrape Google News and insert into google_news_links.
google_news_raw_df = run_fetch_google_news(ticker_list=filtered_tickers, period="12h", workers=20)


# Pull recent article counts for scoring.
google_news_df = run_sql("""

SELECT 
    ticker,
    COUNT(*) AS article_count,
    MAX(created_at) AS latest_article_time
FROM google_news_links
WHERE created_at >= NOW() - INTERVAL 12 HOUR
GROUP BY ticker
ORDER BY article_count DESC;


""").to_df()

news_tickers = google_news_df["ticker"].dropna().unique().tolist() if not google_news_df.empty else []
update_loading_page(driver, f"Complete - {len(news_tickers)} stocks have recent Google News")

##################################################################################################################


In [ ]:
# Final score and save the ranked stock universe.
##################################################################################################################
from universe_scoring import score_stock_universe

stocktwits_signal_df = final_df.copy() if "final_df" in globals() else pd.DataFrame()
news_signal_df = google_news_df.copy() if "google_news_df" in globals() else pd.DataFrame()

scored_stock_universe_df = score_stock_universe(
    fundamentals_df=robinhood_fundamentals_df,
    stocktwits_df=stocktwits_signal_df,
    news_df=news_signal_df,
)

scored_stock_universe_df.to_csv("scored_stock_universe.csv", index=False)

if "driver" in globals():
    update_loading_page(driver, f"Saved scored_stock_universe.csv - {len(scored_stock_universe_df)} rows")

preview_cols = [
    "ticker",
    "signal_score",
    "fundamental_score",
    "stocktwits_score",
    "news_score",
    "open",
    "volume",
    "rel_volume",
    "range_pct",
]

scored_stock_universe_df[[c for c in preview_cols if c in scored_stock_universe_df.columns]].head(50)
##################################################################################################################
